In [4]:
import PyPDF2

In [5]:
import chromadb

In [6]:
import langchain
import chromadb
import PyPDF2

In [7]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("Alice in the wonderland.pdf")  # Replace with the path to your PDF file
documents = loader.load()
print(f"Loaded {len(documents)} documents from the PDF")


Loaded 54 documents from the PDF


In [8]:
# Filter out irrelevant chunks based on specific keywords or page content
filtered_documents = [
    doc for doc in documents 
    if not any(
        keyword in doc.page_content.lower() 
        for keyword in ["copyright", "dedication", "isbn", "all rights reserved"]
    )
]

In [9]:
# Remove irrelevant content based on keywords and chunk size
filtered_chunks = [
    chunk for chunk in chunks
    if len(chunk.page_content.strip()) > 50  # Exclude very small chunks
    and not any(
        keyword in chunk.page_content.lower()
        for keyword in ["dedication", "copyright", "isbn", "reserved", "library"]
    )
]


NameError: name 'chunks' is not defined

In [10]:
#purpose of this class is to split long documents into smaller, manageable chunks of text.
from langchain.text_splitter import RecursiveCharacterTextSplitter


#creating an instance of text splitters
# max chunk== max size of each chunk in characters,
#For example, if a document is 3000 characters long, it will be divided into chunks of about 1000 characters each.

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)

#chunk_overlap=200:
#Adds an overlap of 200 characters between consecutive chunks.
#This ensures some context is preserved between chunks, which is helpful when working with AI models to maintain coherence.

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")


Split into 127 chunks


In [11]:
# Let's create the chroma vector store first with embedding function

from sentence_transformers import SentenceTransformer
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document

In [12]:
from langchain.embeddings.base import Embeddings

# Step 1: Create a custom embedding class
class LocalEmbedding(Embeddings):
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        # Batch encode texts and convert NumPy arrays to lists
        return [embedding.tolist() for embedding in self.model.encode(texts, batch_size=8, show_progress_bar=True)]

    def embed_query(self, text):
        # Encode a single query and convert NumPy array to list
        return self.model.encode(text, show_progress_bar=False).tolist()

# Step 2: Initialize the embedding function with your local model
local_embeddings = LocalEmbedding(model_name="all-MiniLM-L6-v2")


In [13]:
# Step 3: Prepare your data as a list of Document objects
documents = [Document(page_content=chunk.page_content, metadata=chunk.metadata) for chunk in filtered_chunks]

NameError: name 'filtered_chunks' is not defined

In [ ]:
# Step 4: Initialize the Chroma vector store with the embedding function
vector_db = Chroma(embedding_function=local_embeddings, persist_directory="chroma_db")

In [ ]:
# Step 5: Add documents to the Chroma database
vector_db.add_documents(filtered_chunks)

In [1]:
vector_db.persist()
print("Vector database created and saved locally!")

NameError: name 'vector_db' is not defined

In [ ]:
vector_db = Chroma(persist_directory="chroma_db", embedding_function=local_embeddings)


In [ ]:
# k = number of relevant chunks to retrieve
query = "Why did Mariam marry Rasheed in 'A Thousand Splendid Suns'?"
results = vector_db.similarity_search(query, k=3)


In [ ]:
# Each result in results is a Document object with page_content (text of the chunk) and metadata.
for i, result in enumerate(results):
    print(f"Result {i+1}:\n{result.page_content}\n")
